# Build a basic classification model

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
 
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

In [2]:
#  1. Load data
# ------------------------------------------------------------------
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
print(f"Dataset shape: {X.shape}, classes: {list(data.target_names)}\n")
 

Dataset shape: (569, 30), classes: [np.str_('malignant'), np.str_('benign')]



In [3]:
# ------------------------------------------------------------------
# 2. Train / test split
# ------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 

In [4]:
# 3. Define models
#    LR and KNN need scaling (distance / gradient based).
#    Decision tree does not.
# ------------------------------------------------------------------
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
    ),
    "KNN (k=5)": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5)
    ),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
}
 

In [5]:
#  4. Train and evaluate
# ------------------------------------------------------------------
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    cv = cross_val_score(model, X, y, cv=5).mean()
    results.append({"Model": name, "Test Accuracy": acc, "5-Fold CV Accuracy": cv})
 
    print("=" * 60)
    print(name)
    print("=" * 60)
    print("Confusion matrix:\n", confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, target_names=data.target_names))
 
results_df = pd.DataFrame(results).round(4)
print("\nModel comparison:")
print(results_df.to_string(index=False))

Logistic Regression
Confusion matrix:
 [[41  1]
 [ 1 71]]
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

KNN (k=5)
Confusion matrix:
 [[39  3]
 [ 2 70]]
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Decision Tree
Confusion matrix:
 [[39  3]
 [ 4 68]]
              precision    recall  f1-score   support

   malignant       0.91      0.93      0.92        42
      benign       0.96      0.94      0.95        72

    accuracy            

In [6]:
# 5. Feature importance
# ------------------------------------------------------------------
# (a) Logistic Regression: absolute coefficients (features are scaled)
lr = models["Logistic Regression"]
lr_imp = pd.Series(
    np.abs(lr.named_steps["logisticregression"].coef_[0]), index=X.columns
).sort_values(ascending=False)
 
# (b) Decision Tree: Gini-based feature_importances_
dt = models["Decision Tree"]
dt_imp = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)
 
# (c) KNN has no built-in importance -> use permutation importance
knn = models["KNN (k=5)"]
perm = permutation_importance(
    knn, X_test, y_test, n_repeats=20, random_state=42
)
knn_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
 
print("\nTop 5 features - Logistic Regression (|coef|):")
print(lr_imp.head(5).round(4))
print("\nTop 5 features - Decision Tree (Gini importance):")
print(dt_imp.head(5).round(4))
print("\nTop 5 features - KNN (permutation importance):")
print(knn_imp.head(5).round(4))


Top 5 features - Logistic Regression (|coef|):
worst texture           1.2551
radius error            1.0830
worst concave points    0.9537
worst area              0.9478
worst radius            0.9476
dtype: float64

Top 5 features - Decision Tree (Gini importance):
worst radius            0.7335
worst concave points    0.1220
texture error           0.0458
worst texture           0.0323
worst concavity         0.0172
dtype: float64

Top 5 features - KNN (permutation importance):
area error          0.0075
radius error        0.0066
perimeter error     0.0031
worst smoothness    0.0031
worst texture       0.0031
dtype: float64


In [10]:
# 6. Plot
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (title, imp) in zip(
    axes,
    [
        ("Logistic Regression (|coef|)", lr_imp),
        ("KNN (permutation)", knn_imp),
        ("Decision Tree (Gini)", dt_imp),
    ],
):
    top = imp.head(10)[::-1]
    ax.barh(top.index, top.values)
    ax.set_title(title)
    ax.set_xlabel("Importance")
plt.tight_layout()

plt.savefig("feature_importance.png", dpi=120)  # saves in current working folder
print("\nSaved plot: feature_importance.png")
plt.show()  


Saved plot: feature_importance.png


C:\Users\COMPUTER AGE#\AppData\Local\Temp\ipykernel_18416\1614501224.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
